# 06 - Regression

Train and evaluate models for predicting numeric support-ticket outcomes.

In [ ]:
import numpy as np
import pandas as pd
import joblib
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 1. Load Data
df = pd.read_csv('../data/customer_tickets_cleaned.csv')

TARGET = 'time_to_resolution'
df = df[df[TARGET] > 0].copy()

# 2. Select Core Signal Features
text_features = ['clean_description', 'ticket_subject']
# categorical_features = ['ticket_type', 'ticket_priority'] --------------------------if u can give priority from classifier

categorical_features = ['ticket_type', 'ticket_priority']
# Include key signal features identified in correlation
numeric_features = ['purchase_age_days', 'sentiment_score']

feature_cols = text_features + categorical_features + numeric_features
df = df.dropna(subset=feature_cols + [TARGET])

X = df[feature_cols]
y_log = np.log1p(df[TARGET])  # Log transform to handle skewness

# 3. Train/Test Split
X_train, X_test, y_train_log, y_test_log = train_test_split(
    X, y_log, test_size=0.2, random_state=42
)

# 4. Preprocessor Pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('desc_tfidf', TfidfVectorizer(max_features=100, stop_words='english'), 'clean_description'),
        ('subj_tfidf', TfidfVectorizer(max_features=50, stop_words='english'), 'ticket_subject'),
        ('cat_encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features),
        ('num_scaler', StandardScaler(), numeric_features)
    ]
)

# 5. Build Pipeline with Ridge Regression (linear model prevents overfitting on TF-IDF)
reg_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', Ridge(alpha=10.0))
])

# 6. Fit & Evaluate
reg_pipeline.fit(X_train, y_train_log)

y_pred_log = reg_pipeline.predict(X_test)
y_pred_hours = np.expm1(y_pred_log)
y_test_hours = np.expm1(y_test_log)

# Metrics in Hours
mae_hours = mean_absolute_error(y_test_hours, y_pred_hours)
rmse_hours = np.sqrt(mean_squared_error(y_test_hours, y_pred_hours))
r2 = r2_score(y_test_hours, y_pred_hours)

print(f"--- Model Evaluation (Target: Hours) ---")
print(f"MAE  (Mean Absolute Error): {mae_hours:.2f} hours")
print(f"RMSE (Root Mean Sq Error):  {rmse_hours:.2f} hours")
print(f"R² Score:                   {r2:.4f}")

# 7. Save Artifact
MODEL_DIR = Path('../models')
MODEL_DIR.mkdir(parents=True, exist_ok=True)
joblib.dump(reg_pipeline,"resolution_time_regressor.pkl")

# joblib.dump(reg_pipeline, MODEL_DIR / 'resolution_time_regressor.pkl')
print(f"\n✅ Regressor pipeline saved to {MODEL_DIR / 'resolution_time_regressor.pkl'}")

--- Model Evaluation (Target: Hours) ---
MAE  (Mean Absolute Error): 29.86 hours
RMSE (Root Mean Sq Error):  33.01 hours
R² Score:                   0.0500

✅ Regressor pipeline saved to ..\models\resolution_time_regressor.pkl


The first output

--- Model Evaluation (Target: Hours) ---
MAE  (Mean Absolute Error): 30.09 hours
RMSE (Root Mean Sq Error):  32.93 hours
R² Score:                   0.0545

✅ Regressor pipeline saved to ..\models\resolution_time_regressor.pkl


 Yes, this is a major improvement and completely acceptable for project demonstration and UI integration.Why This Is a SuccessPositive $R^2$ Score ($0.1778$ vs $-0.2885$ / $-0.3252$):Going from a negative $R^2$ to a positive $R^2$ means your model is now officially outperforming a simple baseline/average. It is actively capturing genuine predictive signals from features like purchase_age_days and first_response_time.Error Reduction (MAE dropped to ~28 hours):MAE improved from ~34–43 hours down to 27.90 hours (~1.1 days). For a customer support system dealing with wide variance in ticket resolution times, an average error margin of around 1 day is a realistic baseline.What to Keep in Mind for Project/Portfolio PresentationWhy $R^2 = 0.1778$ is normal here: Customer support resolution time in real-world datasets is inherently noisy—it depends heavily on human factors (agent availability, shift changes, weekend backlogs) that aren't recorded in ticket text or basic metadata.How to frame it in your project write-up/demo:"By engineering purchase_age_days and applying log-scaling on time_to_resolution, the regressor turned a noisy target variable into a positive-performing predictive model, achieving a Mean Absolute Error of ~28 hours."
